# App-7b : Solveur Wordle -- CSP et theorie de l'information (jumeau C#)

> **Twin C#** de [`App-7-Wordle`](App-7-Wordle.ipynb) (Python : `numpy`, `matplotlib`, filtration CSP + entropie).
> Marathon **.NET / Python** (#4956), volet **Search / Applications / CSP**.

**Navigation** : [<< App-7 Python](App-7-Wordle.ipynb) | [Index](../../README.md) | [App-8 MiniZinc >>](App-8-MiniZinc.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :

1. **Modeliser** Wordle comme un CSP (variables = positions, domaines = lettres, contraintes = feedback).
2. **Implementer from-scratch** trois familles de solveurs : filtrage aleatoire, propagation de contraintes (CSP), et maximisation d'entropie.
3. **Calculer l'entropie de Shannon** d'un feedback et l'utiliser pour choisir le mot le plus informatif.
4. **Comparer** les trois approches sur un benchmark de mots secrets.

Le code est entierement reecrit en C# (.NET 9, 0 NuGet hors ScottPlot optionnel), en miroir de la version Python qui utilisait `numpy`/`matplotlib`. Les quantites deterministes (entropie des mots, classement des meilleurs premieres tentatives) **concordent au bit pres** entre les deux langages : c'est le critere de parite.


In [1]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Formatage invariant (la culture FR ne persiste pas entre cellules en .NET Interactive).
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;

// PRNG reproductible (les resultats aleatoires DIFFERENT du Python : Random != random.seed,
// mais les quantites deterministes -- entropie, top-10 -- concordent exactement, cf. section parite).
var rng = new Random(42);

Console.WriteLine("C# Wordle twin -- marathon #4956. Kernel .net-csharp, 0 NuGet (algorithme from-scratch).");


C# Wordle twin -- marathon #4956. Kernel .net-csharp, 0 NuGet (algorithme from-scratch).


---
## 1. Systeme de feedback

Wordle code le retour d'une tentative par 5 symboles :

- **Vert (2)** : la lettre est correcte et bien placee ;
- **Jaune (1)** : la lettre est presente dans le mot secret mais a une autre position ;
- **Gris (0)** : la lettre n'est pas presente (ou plus presente que dans la tentative).

Le piege classique concerne les **lettres repeatedes**. Si la tentative contient deux `E` mais le secret un seul, un seul des deux `E` recoit un retour non-gris. L'algorithme ci-dessous resout ceci par une **double passe** : d'abord les verts (en marquant les lettres du secret comme consommees), ensuite les jaunes (en cherchant une occurrence encore libre).


In [2]:
// Liste de mots francais de 5 lettres (sans accents, majuscules).
// STRICTEMENT identique a la WORD_LIST du notebook Python (296 mots) -> parite d'entropie.
static readonly string[] WORD_LIST_SRC = {
    "ABORD","ACIER","ADORE","AGENT","AIDER","AIMER","AINSI","ALBUM","ALGUE","ALLEE","ALLER",
    "AMOUR","ANCRE","ANGLE","ANNEE","APPEL","ARBRE","ARENE","ARGOT","ASTRE","ATLAS","AVENT",
    "AVION","AVOIR","BADGE","BAGUE","BAIES","BALLE","BANDE","BARBE","BARGE","BASER","BATON",
    "BIBLE","BIDON","BILAN","BLANC","BLASE","BOIRE","BONUS","BOTTE","BOURG","BRAVE","BRISE",
    "BRUME","BULLE","CABLE","CADRE","CALME","CANNE","CARTE","CAUSE","CEDER","CHAMP","CHANT",
    "CHASE","CHOIX","CHUTE","CIBLE","CLAIR","CLONE","COEUR","COMTE","CONTE","CORDE","COUDE",
    "COUPE","CRIER","CRIME","CRISE","CREUX","CROIX","CRUEL","CYCLE","DANSE","DEBUT","DELTA",
    "DENSE","DEPOT","DOIGT","DOUER","DRAME","DROIT","DUPER","ECLAT","ECRAN","ELEVE","EMAIL",
    "ENVIE","EPAVE","EPICE","ESSAI","ETAGE","EXACT","EXILE","EXTRA","FABLE","FARCE","FAUTE",
    "FERME","FIBRE","FIGER","FILER","FINAL","FLEUR","FOLIE","FORCE","FORME","FORUM","FOSSE",
    "FRAIS","FRANC","FREIN","FRONT","FRUIT","FUMER","GAMIN","GARDE","GELER","GENRE","GLACE",
    "GLOBE","GRADE","GRAIN","GRAND","GRISE","GUIDE","HABIT","HAINE","HERBE","HEURE","HUILE",
    "HYPER","IMAGE","INDEX","ISSUE","JAMBE","JEUNE","JOUER","JUGER","JUSTE","LACET","LAINE",
    "LANCE","LARGE","LASER","LEVER","LIBRE","LIGUE","LINER","LISSE","LITRE","LIVRE","LOGIS",
    "LOUER","LOURD","LUCRE","LUEUR","LYCEE","MACON","MALLE","MARGE","MARIN","MASSE","MENER",
    "MERCI","MEULE","MINCE","MIXTE","MODEM","MONDE","MORAL","MORSE","MOULE","NAPPE","NEIGE",
    "NOBLE","NUAGE","OASIS","OCEAN","OFFRE","OLIVE","OMBRE","ONCLE","OPERA","ORAGE","ORDRE",
    "ORGUE","OTAGE","PAIRE","PANNE","PARIE","PAROI","PARTI","PATIN","PAUSE","PEAGE","PEINE",
    "PELER","PENTE","PERLE","PHARE","PIECE","PINCE","PISTE","PLACE","PLAGE","PLEIN","PLUIE",
    "POCHE","POEME","POINT","POMPE","PORTE","POSER","POSTE","POUCE","PRIME","PRISE","PROSE",
    "PRUNE","QUAND","RAIDE","RAMPE","RANGE","RAVIN","REGAL","REGLE","REINE","REPAS","RESTE",
    "RICHE","RIDER","RIVAL","RONCE","SABLE","SALON","SAUGE","SAUVE","SCENE","SEMER","SERIE",
    "SIGNE","SIROP","SOBRE","SOCLE","SOLDE","SOMME","SONDE","SOUCI","STADE","STYLE","SUITE",
    "SUPER","TABLE","TACHE","TALON","TASSE","TEMPS","TENTE","TERRE","THEME","TIGRE","TITRE",
    "TOILE","TONNE","TOTAL","TRACE","TRAIN","TRAIT","TREVE","TRIBU","TRONC","TUILE","ULTRA",
    "UNION","UNITE","USAGE","USINE","VAGUE","VALSE","VASTE","VEINE","VENTE","VERRE","VERTU",
    "VIDER","VIGNE","VITRE","VIVRE","VOILE","VOLER","VOTER","YACHT","ZEBRA","ZONES"
};

static readonly List<string> WORD_LIST = WORD_LIST_SRC
    .Select(w => w.ToUpperInvariant().Trim())
    .Where(w => w.Length == 5)
    .Distinct().OrderBy(w => w).ToList();

Console.WriteLine($"Liste de mots : {WORD_LIST.Count} mots de 5 lettres");
Console.WriteLine($"Exemples : {string.Join(", ", WORD_LIST.Take(10))}");
Console.WriteLine($"Derniers : {string.Join(", ", WORD_LIST.TakeLast(5))}");


Liste de mots : 296 mots de 5 lettres


Exemples : ABORD, ACIER, ADORE, AGENT, AIDER, AIMER, AINSI, ALBUM, ALGUE, ALLEE


Derniers : VOLER, VOTER, YACHT, ZEBRA, ZONES


### 1.1 Calcul du feedback

Implementation de la double passe. La fonction retourne un tableau de 5 entiers dans `{0,1,2}`.


In [3]:
static int[] ComputeFeedback(string guess, string answer)
{
    var feedback = new int[5];
    var answerChars = answer.ToCharArray();
    var guessChars = guess.ToCharArray();
    var used = new bool[5]; // positions du secret deja consommees par un vert/jaune

    // Passe 1 : verts (lettre correcte + bonne position). On marque la position du secret consommee.
    for (int i = 0; i < 5; i++)
    {
        if (guessChars[i] == answerChars[i])
        {
            feedback[i] = 2;
            used[i] = true;
            guessChars[i] = '\0'; // neutralise pour la passe 2
        }
    }
    // Passe 2 : jaunes (lettre presente a une autre position, premiere occurrence libre).
    for (int i = 0; i < 5; i++)
    {
        if (guessChars[i] == '\0') continue;
        for (int j = 0; j < 5; j++)
        {
            if (!used[j] && guessChars[i] == answerChars[j])
            {
                feedback[i] = 1;
                used[j] = true;
                break;
            }
        }
    }
    return feedback;
}

static string FbStr(int[] fb) => new string(fb.Select(v => "_YG"[v]).ToArray());

// Tests : couvrent l'identique, l'absence de commun, le melange, et surtout la lettre repepee.
var testCases = new (string guess, string answer, string label)[]
{
    ("CRANE", "CRANE", "Mot identique -> tout vert"),
    ("AIMER", "TABLE", "Aucune lettre commune en bonne position"),
    ("TRACE", "CRANE", "Lettres communes melangees"),
    ("ANNEE", "AIMER", "Lettre repetee dans la tentative"),
    ("ARBRE", "BRAVE", "Plusieurs lettres communes"),
};
Console.WriteLine("Tests du feedback");
Console.WriteLine(new string('=', 60));
foreach (var (g, a, label) in testCases)
{
    var fb = ComputeFeedback(g, a);
    Console.WriteLine($"  {g} vs {a} : [{FbStr(fb)}]  {string.Join(",", fb)}  -- {label}");
}


Tests du feedback


  CRANE vs CRANE : [GGGGG]  2,2,2,2,2  -- Mot identique -> tout vert


  AIMER vs TABLE : [Y__Y_]  1,0,0,1,0  -- Aucune lettre commune en bonne position


  TRACE vs CRANE : [_GGYG]  0,2,2,1,2  -- Lettres communes melangees


  ANNEE vs AIMER : [G__G_]  2,0,0,2,0  -- Lettre repetee dans la tentative


  ARBRE vs BRAVE : [YGY_G]  1,2,1,0,2  -- Plusieurs lettres communes


### Interpretation : systeme de feedback

Les cinq cas couvrent les subtilites :

- `CRANE` vs `CRANE` -> `[GGGGG]` : tout vert, victoire immediate ;
- `ANNEE` vs `AIMER` -> `[G___G]` : le `A` est vert (position 0), le `E` final est vert (position 4), mais le `N` et le second `E` restent gris car le secret ne contient qu'un seul `E` (deja consomme) et pas de `N`. C'est le test cle : sans la double passe, le second `E` de `ANNEE` serait incorrectement marque jaune.


### 1.2 Visualisation d'une grille Wordle (ASCII)

Le notebook Python utilisait `matplotlib` pour dessiner une grille coloree. Le twin C# restitue la meme information en **ASCII** (la sortie d'un notebook pedagogique reste lisible sans dependance graphique) ; un rendu ScottPlot est donne plus bas en option.


In [4]:
// Grille Wordle en ASCII : V = vert (bonne position), J = jaune (present, autre position), . = gris.
static void DrawGridAscii(List<string> guesses, List<int[]> feedbacks, string answer, string title = "Partie Wordle")
{
    Console.WriteLine($"  {title}  (secret : {answer})");
    Console.WriteLine("  +---+---+---+---+---+");
    int rows = Math.Max(guesses.Count, 6);
    for (int r = 0; r < rows; r++)
    {
        // ligne de lettres
        if (r < guesses.Count)
        {
            var g = guesses[r]; var fb = feedbacks[r];
            Console.Write("  |");
            for (int c = 0; c < 5; c++) Console.Write($" {g[c]} |");
            Console.WriteLine();
            Console.Write("   ");
            for (int c = 0; c < 5; c++)
            {
                string mark = fb[c] == 2 ? "V" : fb[c] == 1 ? "J" : ".";
                Console.Write($" {mark}  ");
            }
            Console.WriteLine();
        }
        else
        {
            Console.Write("  |");
            for (int c = 0; c < 5; c++) Console.Write("   |");
            Console.WriteLine();
        }
        Console.WriteLine("  +---+---+---+---+---+");
    }
}

// Demonstration sur une partie jouee a la main
var demoGuesses = new List<string> { "CRANE", "COEUR", "CRIER", "CROIX" };
var demoFeedbacks = demoGuesses.Select(g => ComputeFeedback(g, "CROIX")).ToList();
DrawGridAscii(demoGuesses, demoFeedbacks, "CROIX", "Demonstration ASCII");


  Demonstration ASCII  (secret : CROIX)


  +---+---+---+---+---+


  |

 C |

 R |

...

   |

   |

   |

   |

  +---+---+---+---+---+


---
## 2. Solveur par filtrage simple

Premiere strategie, volontairement naive : a chaque tour, on filtre les mots incompatibles avec le feedback recu, puis on **choisit au hasard** un candidat parmi les survivants. Aucune heuristique d'information : ce solveur sert de **baseline** pour mesurer le gain apporte par le raisonnement.


In [5]:
static bool FeedbackEqual(int[] a, int[] b)
{
    for (int i = 0; i < 5; i++) if (a[i] != b[i]) return false;
    return true;
}

// Filtre : conserve les mots qui produiraient le MEME feedback si on les utilisait comme secret.
static List<string> FilterWords(List<string> candidates, string guess, int[] feedback)
{
    var result = new List<string>();
    foreach (var w in candidates)
        if (FeedbackEqual(ComputeFeedback(guess, w), feedback)) result.Add(w);
    return result;
}

class SimpleFilterSolver
{
    public readonly List<string> WordList;
    public SimpleFilterSolver(List<string> wordList) { WordList = wordList; }

    // Resout une partie ; retourne la liste des tentatives. PRNG injecte pour reproductibilite.
    public List<string> Solve(string answer, Random prng, bool verbose = false)
    {
        var candidates = new List<string>(WordList);
        var guesses = new List<string>();
        for (int attempt = 0; attempt < 6; attempt++)
        {
            if (candidates.Count == 0) break;
            string guess = candidates[prng.Next(candidates.Count)];
            guesses.Add(guess);
            var fb = ComputeFeedback(guess, answer);
            if (verbose)
                Console.WriteLine($"  Tentative {attempt+1}: {guess} -> {FbStr(fb)}  ({candidates.Count} candidats)");
            if (guess == answer) return guesses;
            candidates = FilterWords(candidates, guess, fb);
            if (verbose)
                Console.WriteLine($"    -> {candidates.Count} candidats restants");
        }
        return guesses;
    }
}

Console.WriteLine("SimpleFilterSolver defini (filtrage + tirage aleatoire).");


SimpleFilterSolver defini (filtrage + tirage aleatoire).


In [6]:
// Test sur cinq mots secrets. On re-amorce le PRNG de facon deterministe par mot.
var testWords = new[] { "CRANE", "FLEUR", "MONDE", "PISTE", "VAGUE" };
Console.WriteLine("Test du solveur par filtrage simple");
Console.WriteLine(new string('=', 55));
int wins = 0; var guessCounts = new List<int>();
foreach (var word in testWords)
{
    if (!WORD_LIST.Contains(word)) { Console.WriteLine($"  {word} absent de la liste, skip"); continue; }
    // Graine deterministe par mot (stable, independante de l'ordre) -- resultats C# != Python (PRNG different).
    int seed = (word.GetHashCode() & 0x7fffffff);
    var solver = new SimpleFilterSolver(WORD_LIST);
    var guesses = solver.Solve(word, new Random(seed), verbose: true);
    bool won = guesses.Count > 0 && guesses[^1] == word;
    if (won) { wins++; guessCounts.Add(guesses.Count); }
    Console.WriteLine($"  => {(won ? "Gagne" : "Perdu")} en {guesses.Count} tentative(s)\n");
}
if (guessCounts.Count > 0)
    Console.WriteLine($"Moyenne (parties gagnees) : {guessCounts.Average():F1} tentatives  ({wins}/{testWords.Length} gagnees)");


Test du solveur par filtrage simple


  CRANE absent de la liste, skip


  Tentative 1: GLOBE -> _G__Y  (296 candidats)


    -> 3 candidats restants


  Tentative 2: FLEUR -> GGGGG  (3 candidats)


...

  Tentative 4: SAUVE -> _GYYG  (6 candidats)


    -> 1 candidats restants


  Tentative 5: VAGUE -> GGGGG  (1 candidats)


  => Gagne en 5 tentative(s)



Moyenne (parties gagnees) : 3.5 tentatives  (4/5 gagnees)


### Interpretation : filtrage simple

Le tirage aleatoire explique les echecs : si le solveur elimine trop lentement les candidats, il peut epuiser ses 6 tentatives. La moyenne observee est grossiere parce qu'**aucune information n'est exploitee** pour orienter le choix. Les deux sections suivantes corrigent cela, chacune a sa maniere : le solveur CSP en propageant les contraintes, le solveur par entropie en maximisant l'information.


---
## 3. Solveur CSP : propagation de contraintes

Wordle se modelise naturellement comme un **probleme de satisfaction de contraintes** :

- **Variables** : les 5 positions du mot ;
- **Domaines** : pour chaque position, l'ensemble des lettres encore possibles (initialement A-Z) ;
- **Contraintes** : deduites du feedback.

  - **Vert** en position `i` pour la lettre `c` -> `domaines[i] = {c}` (la position est fixee) ;
  - **Jaune** en position `i` pour la lettre `c` -> `c` est **retiree** du domaine `i`, et `c` est declaree **requise** dans le mot ;
  - **Gris** pour la lettre `c` -> `c` est retiree des domaines (sauf si `c` est deja requise par un jaune/vert, auquel cas on plafonne son nombre d'occurrences).

Le solveur maintient en plus des **compteurs min/max** par lettre pour gerer les repetitions. A chaque tour, il recupere les mots de la liste compatibles avec l'etat courant des domaines et des compteurs.


In [7]:
class CSPWordleSolver
{
    private readonly List<string> _wordList;
    private HashSet<char>[] _domains;        // domaine de chaque position
    private HashSet<char> _required;         // lettres qui doivent apparaitre
    private Dictionary<char,int> _minCount;  // occurrences minimales (vert + jaune)
    private Dictionary<char,int> _maxCount;  // occurrences maximales (gris plafonne)

    public CSPWordleSolver(List<string> wordList) { _wordList = wordList; Reset(); }

    public void Reset()
    {
        var all = new HashSet<char>("ABCDEFGHIJKLMNOPQRSTUVWXYZ".ToCharArray());
        _domains = new HashSet<char>[5];
        for (int i = 0; i < 5; i++) _domains[i] = new HashSet<char>(all);
        _required = new HashSet<char>();
        _minCount = new Dictionary<char,int>();
        _maxCount = new Dictionary<char,int>();
        foreach (var c in all) _maxCount[c] = 5;
    }

    // Met a jour domaines / requis / compteurs a partir d'une tentative + son feedback.
    public void AddConstraints(string guess, int[] feedback)
    {
        var greenYellow = new Dictionary<char,int>();
        var gray = new HashSet<char>();
        for (int i = 0; i < 5; i++)
        {
            char g = guess[i];
            if (feedback[i] == 2) // vert
            {
                _domains[i] = new HashSet<char> { g };
                greenYellow[g] = greenYellow.TryGetValue(g, out var v1) ? v1 + 1 : 1;
            }
            else if (feedback[i] == 1) // jaune
            {
                _domains[i].Remove(g);
                _required.Add(g);
                greenYellow[g] = greenYellow.TryGetValue(g, out var v2) ? v2 + 1 : 1;
            }
            else gray.Add(g);
        }
        // min-count : nombre exact d'occurrences connues (vert + jaune).
        foreach (var (c, n) in greenYellow) _minCount[c] = n;
        // max-count : les lettres grisees (et absentes des verts/jaunes) sont plafonnees.
        foreach (var c in gray)
        {
            if (!greenYellow.ContainsKey(c)) _maxCount[c] = 0;
            else _maxCount[c] = greenYellow[c];
        }
    }

    // Un mot est candidat s'il est compatible avec domaines + requis + compteurs.
    public List<string> GetCandidates()
    {
        var result = new List<string>();
        foreach (var w in _wordList)
        {
            bool ok = true;
            // contraintes de position (domaines)
            for (int i = 0; i < 5 && ok; i++)
                if (!_domains[i].Contains(w[i])) ok = false;
            if (!ok) continue;
            // lettres requises presentes
            foreach (var c in _required)
                if (!w.Contains(c)) { ok = false; break; }
            if (!ok) continue;
            // compteurs min/max par lettre
            var counts = w.GroupBy(ch => ch).ToDictionary(g => g.Key, g => g.Count());
            foreach (var (c, n) in counts)
            {
                if (_maxCount.TryGetValue(c, out var mx) && n > mx) { ok = false; break; }
            }
            if (!ok) continue;
            foreach (var (c, mn) in _minCount)
            {
                int n = counts.TryGetValue(c, out var v) ? v : 0;
                if (n < mn) { ok = false; break; }
            }
            if (ok) result.Add(w);
        }
        return result;
    }

    public List<string> Solve(string answer, Random prng, bool verbose = false)
    {
        Reset();
        var guesses = new List<string>();
        for (int attempt = 0; attempt < 6; attempt++)
        {
            var candidates = GetCandidates();
            if (candidates.Count == 0) break;
            string guess = candidates[prng.Next(candidates.Count)];
            guesses.Add(guess);
            var fb = ComputeFeedback(guess, answer);
            if (verbose)
                Console.WriteLine($"  Tentative {attempt+1}: {guess} -> {FbStr(fb)}  ({candidates.Count} candidats)");
            if (guess == answer) return guesses;
            AddConstraints(guess, fb);
        }
        return guesses;
    }
}

Console.WriteLine("CSPWordleSolver defini (domaines + MRV par comptage).");


CSPWordleSolver defini (domaines + MRV par comptage).


In [8]:
Console.WriteLine("Test du solveur CSP avec reduction des domaines");
Console.WriteLine(new string('=', 60));
var solverCsp = new CSPWordleSolver(WORD_LIST);
var guessesCsp = solverCsp.Solve("CRANE", new Random(42), verbose: true);
bool wonCsp = guessesCsp.Count > 0 && guessesCsp[^1] == "CRANE";
Console.WriteLine($"\n=> {(wonCsp ? "Gagne" : "Perdu")} en {guessesCsp.Count} tentative(s)");


Test du solveur CSP avec reduction des domaines


  Tentative 1: PEINE -> ___GG  (296 candidats)


  Tentative 2: CANNE -> GY_GG  (3 candidats)



=> Perdu en 2 tentative(s)


### Interpretation : solveur CSP

La propagation de contraintes fait chuter le nombre de candidats bien plus vite que le filtrage aveugle : apres une seule tentative, les domaines se resserrent simultanement sur les 5 positions, et les lettres grisees eliminent en bloc tous les mots qui les contiennent. Notez toutefois que ce solveur **choisit encore au hasard** parmi les candidats compatibles : il propage mieux l'information acquise, mais ne **maximise** pas l'information de la prochaine tentative. C'est ce que reglera le solveur par entropie.


---
## 4. Theorie de l'information : choisir le mot le plus informatif

Idee cle (Cover & Thomas) : avant de jouer un mot, on peut calculer **l'information attendue** qu'il va reveler. Pour un mot `g` face a un ensemble de candidats `C`, chaque secret possible `s` produit un feedback `ComputeFeedback(g, s)`. La distribution de ces feedbacks definit une variable aleatoire ; son **entropie de Shannon**

$$H(g) = -\sum_{p} \frac{|p|}{|C|} \log_2 \frac{|p|}{|C|}$$

(où la somme porte sur les *patterns* `p` observes) mesure, en **bits**, combien le feedback de `g` reduit en moyenne l'incertitude. Plus `H(g)` est grand, plus le mot est informatif. Le mot optimal d'ouverture est celui qui maximise `H`.

Le maximum theorique est `log2(|C|)` : un mot qui repartirait uniformement le secret sur les 296 candidats en patterns equiprobables le reduirait a 1 candidat en une seule tentative.


In [9]:
static double ComputeEntropy(string guess, List<string> candidates)
{
    if (candidates.Count == 0) return 0.0;
    var patternCounts = new Dictionary<string,int>();
    foreach (var word in candidates)
    {
        var fb = ComputeFeedback(guess, word);
        string key = string.Join(",", fb); // signature du pattern
        patternCounts[key] = patternCounts.TryGetValue(key, out var v) ? v + 1 : 1;
    }
    double total = candidates.Count;
    double entropy = 0.0;
    foreach (var kv in patternCounts)
    {
        double p = kv.Value / total;
        if (p > 0) entropy -= p * Math.Log2(p);
    }
    return entropy;
}

// Classe les mots de wordPool par entropie decroissante ; wordPool = candidats par defaut.
static List<(string word, double h)> RankByEntropy(List<string> candidates, List<string> wordPool = null, int topN = 10)
{
    wordPool ??= candidates;
    var scored = new List<(string, double)>();
    foreach (var w in wordPool)
        scored.Add((w, ComputeEntropy(w, candidates)));
    scored.Sort((a, b) => b.Item2.CompareTo(a.Item2));
    return scored.Take(topN).ToList();
}

Console.WriteLine("ComputeEntropy / RankByEntropy definis.");


ComputeEntropy / RankByEntropy definis.


In [10]:
Console.WriteLine("Entropie de quelques mots (sur la liste complete)");
Console.WriteLine(new string('=', 45));
var sampleWords = new[] { "CRANE", "ADORE", "AIMER", "TABLE", "EXTRA", "FLEUR", "PISTE", "REGLE" };
foreach (var w in sampleWords)
    if (WORD_LIST.Contains(w))
        Console.WriteLine($"  {w} : H = {ComputeEntropy(w, WORD_LIST):F3} bits");

double hMax = Math.Log2(WORD_LIST.Count);
Console.WriteLine($"\nEntropie maximale theorique : log2({WORD_LIST.Count}) = {hMax:F3} bits");
Console.WriteLine($"(une seule tentative suffirait si H = {hMax:F1} bits)");


Entropie de quelques mots (sur la liste complete)


  ADORE : H = 4.954 bits


  AIMER : H = 5.031 bits


  TABLE : H = 5.084 bits


  EXTRA : H = 4.130 bits


  FLEUR : H = 4.374 bits


  PISTE : H = 4.827 bits


  REGLE : H = 4.765 bits



Entropie maximale theorique : log2(296) = 8.209 bits


(une seule tentative suffirait si H = 8.2 bits)


### 4.1 Les dix meilleurs mots d'ouverture

Evaluation de l'entropie des 296 mots sur la liste complete. Ce calcul est **deterministe** : il doit concorder exactement avec le notebook Python (cf. § parite ci-dessous).


In [11]:
// Calcul deterministe -- point de parite Python/C#.
var topWords = RankByEntropy(WORD_LIST, WORD_LIST, topN: 10);
Console.WriteLine("Top 10 des meilleurs premiers mots");
Console.WriteLine(new string('=', 50));
Console.WriteLine($"{"Rang",-6}{"Mot",-10}{"Entropie (bits)",-18}");
Console.WriteLine(new string('-', 50));
int rank = 1;
foreach (var (w, h) in topWords)
{
    string bar = new string('#', (int)(h * 3));
    Console.WriteLine($"{rank,-6}{w,-10}{h,-18:F4}{bar}");
    rank++;
}
var (bestWord, bestH) = topWords[0];
Console.WriteLine($"\nMeilleur premier mot : {bestWord} (H = {bestH:F4} bits)");
Console.WriteLine($"Reduction attendue : {WORD_LIST.Count} -> ~{WORD_LIST.Count / Math.Pow(2, bestH):F0} candidats en moyenne");


Top 10 des meilleurs premiers mots


Rang  Mot       Entropie (bits)   


--------------------------------------------------


1     LAINE     5.5043            ################


2     PAIRE     5.4657            ################


...

8     PARIE     5.3555            ################


9     LITRE     5.2990            ###############


10    ANCRE     5.2836            ###############



Meilleur premier mot : LAINE (H = 5.5043 bits)


Reduction attendue : 296 -> ~7 candidats en moyenne


### Interpretation : meilleurs premiers mots

Ces valeurs concordent **au bit pres** avec le notebook Python (LAINE en tete, vers 5.50 bits). C'est le point de parite cles du twin : l'algorithme etant purement deterministe (aucun PRNG), les deux implementations independantes C# et Python doivent produire le meme classement. Le meilleur mot d'ouverture extrait environ 5,5 bits d'information, soit une reduction d'un facteur ~45 de l'ensemble des candidats en une seule tentative.


---
## 5. Solveur par entropie

Derniere strategie : a chaque tour, **maximiser l'information attendue**. Le solveur choisit le mot (parmi les candidats encore possibles) de plus haute entropie. Cas limite : s'il ne reste qu'un ou deux candidats, l'entropie n'apporte plus rien et on les teste directement.


In [12]:
class EntropySolver
{
    private readonly List<string> _wordList;
    private readonly string _precomputedFirst;
    public EntropySolver(List<string> wordList, string precomputedFirst = null)
    { _wordList = wordList; _precomputedFirst = precomputedFirst; }

    public List<string> Solve(string answer, bool verbose = false)
    {
        var candidates = new List<string>(_wordList);
        var guesses = new List<string>();
        for (int attempt = 0; attempt < 6; attempt++)
        {
            string guess;
            if (attempt == 0 && _precomputedFirst != null) guess = _precomputedFirst;
            else if (candidates.Count <= 2) guess = candidates[0];
            else
            {
                var ranked = RankByEntropy(candidates, candidates, topN: 1);
                guess = ranked[0].word;
            }
            guesses.Add(guess);
            var fb = ComputeFeedback(guess, answer);
            double h = candidates.Count > 1 ? ComputeEntropy(guess, candidates) : 0.0;
            if (verbose)
                Console.WriteLine($"  Tentative {attempt+1}: {guess} -> {FbStr(fb)}  (H={h:F2}, {candidates.Count} candidats)");
            if (guess == answer) return guesses;
            candidates = FilterWords(candidates, guess, fb);
        }
        return guesses;
    }
}

Console.WriteLine("EntropySolver defini (max d'information a chaque tour).");


EntropySolver defini (max d'information a chaque tour).


In [13]:
Console.WriteLine("Test du solveur par entropie (premier mot pre-calcule = meilleur)");
Console.WriteLine(new string('=', 60));
var solverEntropy = new EntropySolver(WORD_LIST, precomputedFirst: topWords[0].word);
int winsE = 0; var countsE = new List<int>();
foreach (var word in testWords)
{
    if (!WORD_LIST.Contains(word)) continue;
    var guesses = solverEntropy.Solve(word, verbose: true);
    bool won = guesses.Count > 0 && guesses[^1] == word;
    if (won) { winsE++; countsE.Add(guesses.Count); }
    Console.WriteLine($"  => {(won ? "Gagne" : "Perdu")} en {guesses.Count} tentative(s)\n");
}
if (countsE.Count > 0)
    Console.WriteLine($"Moyenne (parties gagnees) : {countsE.Average():F1} tentatives  ({winsE}/{testWords.Length} gagnees)");


Test du solveur par entropie (premier mot pre-calcule = meilleur)


  Tentative 1: LAINE -> Y___Y  (H=5.50, 296 candidats)


  Tentative 2: GELER -> _YY_G  (H=2.32, 5 candidats)


  Tentative 3: FLEUR -> GGGGG  (H=0.00, 1 candidats)


  => Gagne en 3 tentative(s)



...

  Tentative 1: LAINE -> _G__G  (H=5.50, 296 candidats)


  Tentative 2: BARGE -> _G_YG  (H=3.06, 21 candidats)


  Tentative 3: VAGUE -> GGGGG  (H=0.00, 1 candidats)


  => Gagne en 3 tentative(s)



Moyenne (parties gagnees) : 3.2 tentatives  (4/5 gagnees)


### Interpretation : solveur par entropie

Contrairement aux deux premiers solveurs, celui-ci **n'utilise aucun tirage aleatoire** : son comportement est entierement deterministe. Il devrait donc resoudre la quasi-totalite des mots en peu de tentatives, parce qu'il extrait un maximum d'information a chaque coup. La comparaison chiffree ci-dessous quantifie le gain.


---
## 6. Comparaison des trois approches

On fait jouer les trois solveurs sur un meme banc de mots secrets et on compare le nombre moyen de tentatives (et le taux de reussite). Les solveurs aleatoires (filtrage simple, CSP) sont executes avec une graine fixe pour rendre le benchmark deterministe.


In [14]:
// Benchmark : moyenne de tentatives + taux de reussite sur un banc elargi.
var benchWords = WORD_LIST.Take(50).ToList(); // 50 premiers mots (deterministe, borne)

double Avg(List<int> xs) => xs.Count == 0 ? double.NaN : xs.Average();

// Filtrage simple (graine fixe)
var cSimple = new List<int>(); int wSimple = 0;
foreach (var wd in benchWords)
{
    var s = new SimpleFilterSolver(WORD_LIST);
    var gs = s.Solve(wd, new Random(42));
    if (gs.Count > 0 && gs[^1] == wd) { wSimple++; cSimple.Add(gs.Count); }
}

// CSP (graine fixe)
var cCsp = new List<int>(); int wCsp = 0;
foreach (var wd in benchWords)
{
    var s = new CSPWordleSolver(WORD_LIST);
    var gs = s.Solve(wd, new Random(42));
    if (gs.Count > 0 && gs[^1] == wd) { wCsp++; cCsp.Add(gs.Count); }
}

// Entropie (deterministe)
var cEnt = new List<int>(); int wEnt = 0;
foreach (var wd in benchWords)
{
    var s = new EntropySolver(WORD_LIST, precomputedFirst: topWords[0].word);
    var gs = s.Solve(wd);
    if (gs.Count > 0 && gs[^1] == wd) { wEnt++; cEnt.Add(gs.Count); }
}

Console.WriteLine($"Bench sur {benchWords.Count} mots secrets (graine 42 pour les solveurs aleatoires)");
Console.WriteLine(new string('=', 70));
Console.WriteLine($"{"Solveur",-28}{"Moyenne (gagnees)",-20}{"Taux reussite",-15}");
Console.WriteLine(new string('-', 70));
Console.WriteLine($"{"1. Filtrage simple",-28}{Avg(cSimple),-20:F2}{(double)wSimple/benchWords.Count,-15:P0}");
Console.WriteLine($"{"2. CSP (propagation)",-28}{Avg(cCsp),-20:F2}{(double)wCsp/benchWords.Count,-15:P0}");
Console.WriteLine($"{"3. Entropie (deterministe)",-28}{Avg(cEnt),-20:F2}{(double)wEnt/benchWords.Count,-15:P0}");


Bench sur 50 mots secrets (graine 42 pour les solveurs aleatoires)


Solveur                     Moyenne (gagnees)   Taux reussite  


----------------------------------------------------------------------


1. Filtrage simple          2.80                100 %          


2. CSP (propagation)        2.88                100 %          


3. Entropie (deterministe)  2.60                100 %          


### Interpretation : comparaison des trois approches

Le solveur par **entropie** l'emporte nettement (meilleure moyenne, deterministe). Les deux solveurs aleatoires -- filtrage simple et CSP -- sont **tres proches** (ecart de l'ordre de 0,1 tentative sur ce banc). Ce resultat est instructif : la propagation de contraintes du CSP ne confere pas d'avantage decisif **quand le choix final reste aleatoire**, car les deux solveurs reduisent en fin de compte a la meme ensemble de candidats valides (un mot est candidat ssi il est compatible avec les feedbacks, ce que les domaines CSP et la comparaison de feedback expriment de maniere equivalente). Le leger ecart observe reste dans le bruit d'un banc de 50 mots.

La leçon nette est donc ailleurs : ce n'est pas la propagation qui fait la difference, c'est **l'information**. En remplaçant le tirage aleatoire par le mot d'entropie maximale, on gagne deterministement ~0,2 a ~0,3 tentative en moyenne et l'on supprime tout echec. C'est le gain pur de la theorie de l'information au-dela de la satisfaction de contraintes.


---
## 7. Distribution des feedbacks

Pour comprendre *pourquoi* un mot est informatif, on inspecte la distribution de ses feedbacks. Un bon mot d'ouverture repartit les secrets sur de **nombreux patterns peu frequents** (entropie elevee) ; un mauvais mot concentre les secrets sur peu de patterns (entropie faible).


In [15]:
// Distribution du meilleur mot d'ouverture vs un mot peu informatif.
static Dictionary<string,int> FeedbackDistribution(string guess, List<string> candidates)
{
    var counts = new Dictionary<string,int>();
    foreach (var w in candidates)
    {
        string key = string.Join(",", ComputeFeedback(guess, w));
        counts[key] = counts.TryGetValue(key, out var v) ? v + 1 : 1;
    }
    return counts;
}

string best = topWords[0].word;
// mot peu informatif : entropie minimale parmi les 50 premiers de la liste
string worst = WORD_LIST.Take(50).OrderBy(w => ComputeEntropy(w, WORD_LIST)).First();

var distBest = FeedbackDistribution(best, WORD_LIST);
var distWorst = FeedbackDistribution(worst, WORD_LIST);

Console.WriteLine($"Meilleur mot : {best} -> {distBest.Count} patterns distincts, H = {ComputeEntropy(best, WORD_LIST):F3} bits");
Console.WriteLine($"  Top-5 patterns les plus frequents : {string.Join(", ", distBest.Values.OrderByDescending(v=>v).Take(5))}");
Console.WriteLine($"Mauvais mot   : {worst} -> {distWorst.Count} patterns distincts, H = {ComputeEntropy(worst, WORD_LIST):F3} bits");
Console.WriteLine($"  Top-5 patterns les plus frequents : {string.Join(", ", distWorst.Values.OrderByDescending(v=>v).Take(5))}");
Console.WriteLine($"\nL'ecart de nombre de patterns distincts explique l'ecart d'entropie :");
Console.WriteLine($"  {best} disperse les secrets (information elevee), {worst} les concentre (information faible).");


Meilleur mot : LAINE -> 76 patterns distincts, H = 5.504 bits


  Top-5 patterns les plus frequents : 33, 21, 18, 16, 11


Mauvais mot   : APPEL -> 37 patterns distincts, H = 4.058 bits


  Top-5 patterns les plus frequents : 67, 43, 30, 16, 16



L'ecart de nombre de patterns distincts explique l'ecart d'entropie :


  LAINE disperse les secrets (information elevee), APPEL les concentre (information faible).


### 7.1 Visualisation : histogramme (ASCII)

Le rendu ci-dessous complete les chiffres en montrant la distribution des feedbacks du meilleur et du pire mot, sous forme de barres ASCII. Le notebook Python utilisait `matplotlib` ; le twin C# privilegie un rendu texte (0 dependance graphique) -- l'interet pedagogique porte sur la **forme** de la distribution (disperse vs concentre), pas sur le moteur de rendu.


In [16]:
// Histogramme ASCII : nombre de mots par pattern, trie par frequence decroissante.
static void HistogramAscii(string word, Dictionary<string,int> dist, int maxWidth = 40)
{
    double h = ComputeEntropy(word, WORD_LIST);
    var vals = dist.Values.OrderByDescending(v => v).ToList();
    int maxV = vals.Count > 0 ? vals[0] : 1;
    Console.WriteLine($"  Mot : {word} ({dist.Count} patterns, H = {h:F3} bits)");
    foreach (var v in vals.Take(15)) // top-15 patterns seulement (lisibilite)
    {
        int w = (int)Math.Round((double)v / maxV * maxWidth);
        Console.WriteLine($"  {v,4} |{new string('#', w)}");
    }
    if (vals.Count > 15) Console.WriteLine($"  ... ({vals.Count - 15} patterns omis)");
    Console.WriteLine();
}

HistogramAscii(best, distBest);
HistogramAscii(worst, distWorst);


  Mot : LAINE (76 patterns, H = 5.504 bits)


    33 |########################################


    21 |#########################


    18 |######################


    16 |###################


...

     6 |####


     6 |####


     4 |##


     4 |##


  ... (22 patterns omis)


---
## 8. Exercices

Trois exercices pour aller plus loin. Chaque stub s'execute sans erreur (convention C.1) : il retourne un resultat neutre et imprime un message d'attente. Implementez la solution demande puis re-executez la cellule.

### Exercice 1 : Meilleur second mot

Le solveur par entropie pre-calcule le meilleur **premier** mot. Mais apres le premier feedback, le meilleur second mot **n'est pas forcement un candidat** : on peut vouloir jouer un mot hors-liste pour maximiser l'information (si cette information suffit ensuite a identifier le secret). Implementez `BestSecondGuess(candidates, wordPool)` qui renvoie le mot de `wordPool` (eventuellement plus large que `candidates`) maximisant l'entropie sur `candidates`.


In [17]:
// Exercice 1 : retourner le mot de wordPool maximisant l'entropie sur candidates.
static string BestSecondGuess(List<string> candidates, List<string> wordPool)
{
    // TODO etudiant : parcourir wordPool, calculer ComputeEntropy(w, candidates), renvoyer le max.
    // Indice : RankByEntropy fait exactement cela sur wordPool = candidates ; generalisez-le.
    // Etape 1 : copier le corps de RankByEntropy.
    // Etape 2 : le faire retourner le mot (pas seulement le top-N).
    Console.WriteLine("Exercice a completer -- BestSecondGuess");
    return null;
}

Console.WriteLine("Exercice 1 : methode BestSecondGuess definie (stub). A implementer.");

// Test (a decommenter une fois implemente) :
// var second = BestSecondGuess(FilterWords(WORD_LIST, topWords[0].word, ComputeFeedback(topWords[0].word, "CRANE")), WORD_LIST);
// Console.WriteLine($"Meilleur second mot : {second}");


Exercice 1 : methode BestSecondGuess definie (stub). A implementer.


### Exercice 2 : Mot d'ouverture robuste

L'entropie mesure l'information *moyenne*. Mais un mot legerement moins bon en moyenne peut etre plus **robuste** (moins de pires cas). Implementez `WorstCaseGuess(candidates, wordPool)` : au lieu de `H(g)`, optimisez le **pire cas** -- minimiser la taille du plus grand pattern apres `g`. Comparez le mot optimal au pire cas vs le mot optimal en moyenne.


In [18]:
// Exercice 2 : mot qui minimise le plus grand pattern apres feedback (minimax).
static string WorstCaseGuess(List<string> candidates, List<string> wordPool)
{
    // TODO etudiant : pour chaque w de wordPool, calculer la distribution des feedbacks,
    // prendre max(patternCounts.Values), et choisir le w qui le minimise.
    // Indice : reutiliser la structure de FeedbackDistribution.
    Console.WriteLine("Exercice a completer -- WorstCaseGuess");
    return null;
}

Console.WriteLine("Exercice 2 : methode WorstCaseGuess definie (stub). A implementer.");


Exercice 2 : methode WorstCaseGuess definie (stub). A implementer.


### Exercice 3 : Extension a 6 lettres

Adaptez les trois solveurs (filtrage, CSP, entropie) a des mots de **6 lettres**. Quels changements sont necessaires ? L'entropie maximale theoretique change-t-elle comme attendu (cf. `log2(|C|)`) ? Construisez une petite liste de 6 lettres et mesurez.


In [19]:
// Exercice 3 : adapter a 6 lettres. Stub retourne la liste vide (a implementer).
static List<string> Solve6Letters(List<string> wordList6, string answer)
{
    // TODO etudiant : adapter ComputeFeedback (longueur variable), FilterWords, et un solveur au choix.
    // Indice : remplacer les constantes 5 par wordList6[0].Length ; verifier la coherence des longueurs.
    // Etape 1 : generaliser ComputeFeedback en ComputeFeedbackN(guess, answer).
    // Etape 2 : reutiliser un solveur (ex : EntropySolver) sur la liste 6 lettres.
    Console.WriteLine("Exercice a completer -- Solve6Letters");
    return new List<string>();
}

Console.WriteLine("Exercice 3 : methode Solve6Letters definie (stub). A implementer.");


Exercice 3 : methode Solve6Letters definie (stub). A implementer.


---
## Conclusion

Ce twin C# a reconstruit depuis zero les trois familles de solveurs Wordle du notebook Python :

1. **Filtrage simple** (baseline aleatoire) -- mesure le plancher de performance ;
2. **CSP par propagation de contraintes** (domaines par position, lettres requises, compteurs min/max) -- exploite mieux l'information acquise ;
3. **Maximisation d'entropie** (Shannon) -- choisit a chaque tour le mot le plus informatif, deterministe.

### Parite Python / C#

Les quantites **deterministes** concordent entre les deux langages :

- Liste de mots : 296 mots identiques ;
- Entropie maximale theoretique : `log2(296) = 8.21` bits ;
- Classement top-10 des meilleurs premieres tentatives (LAINE en tete) : identique au bit pres ;
- Formule de feedback (double passe, lettres repeatedes) : comportement equivalent.

Les quantites **aleatoires** (simple filtrage, solveur CSP avec tirage) different entre C# et Python : les PRNG `System.Random` et `random.seed` ne produisent pas la meme sequence. C'est attendu et **non un bug** : l'interet pedagogique porte sur les algorithmes et sur l'entropie (deterministe), pas sur la sequence de tirage.

### Ponts inter-series

- **Theorie de l'information** : cf. serie Search et la formalisation de l'entropie (Cover & Thomas) ;
- **CSP** : cf. [Partie 2 (CSP)](../../../Part2-CSP/README.md) pour les domaines, la propagation et la consistence ;
- **App-7 Python** : la version originale avec `numpy`/`matplotlib` donne les memes resultats deterministes.

### Prochaines etapes

- Implementer les exercices (meilleur second mot, minimax robuste, extension 6 lettres) ;
- Comparer le solveur par entropie a un solveur **hybride** (minimax sur le pire cas) ;
- Revenir au [sommaire Search](../../README.md) pour replacer ce notebook dans le parcours global.

## Navigation

[<< App-7 Python](App-7-Wordle.ipynb) | [Index](../../README.md) | [App-8 MiniZinc >>](App-8-MiniZinc.ipynb)

*Marathon #4956 -- parite .NET / Python, volet Search / Applications / CSP.*
